# Pipeline ResNet18 (CBIS-DDSM)

Clasificación binaria **BENIGNO vs MALIGNO** en imágenes de mamografía completa.

## Antes de ejecutar
1. Generar manifiestos desde CSV + JPEG: `python scripts/build_resnet18_manifest.py`
2. Rutas centralizadas en `scripts/paths.py`

## Datos de entrada
- `src/data/processed/manifest_train.csv`
- `src/data/processed/manifest_test.csv`

## Flujo de este notebook (celdas siguientes)
1. **Imports y semilla** — librerías, `DEVICE` (CUDA si existe)
2. **Carga y split** — train/val desde el manifest de entrenamiento, **agrupado por `patient_id`** para limitar fuga entre cortes del mismo paciente
3. **Dataset y DataLoaders** — `MammographyDataset`, augmentación solo en train, normalización ImageNet
4. **Modelo** — `ResNet18` con pesos ImageNet (`ResNet18_Weights.DEFAULT`), cabeza `fc` con 2 salidas
5. **Entrenamiento** — una época de train y validación por iteración del bucle externo
6. **Test** — cargar mejor checkpoint y métricas finales


In [1]:
# Imports: PyTorch + torchvision (ResNet18), sklearn para split y métricas.
# Semilla fija para reproducibilidad (numpy, random, torch, CUDA).
from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [2]:
# Raíz del repo: si el cwd es la carpeta notebooks/, sube un nivel.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
TRAIN_CSV = ROOT / "src/data/processed/manifest_train.csv"
TEST_CSV = ROOT / "src/data/processed/manifest_test.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# Validación: split por PACIENTE (no por filas sueltas) para no mezclar imágenes del mismo caso.
patients = train_df["patient_id"].dropna().unique()
train_pat, val_pat = train_test_split(patients, test_size=0.2, random_state=SEED)

tr_df = train_df[train_df["patient_id"].isin(train_pat)].copy()
val_df = train_df[train_df["patient_id"].isin(val_pat)].copy()

print("Train:", tr_df.shape, tr_df["label_name"].value_counts().to_dict())
print("Val:", val_df.shape, val_df["label_name"].value_counts().to_dict())
print("Test:", test_df.shape, test_df["label_name"].value_counts().to_dict())

Train: (2315, 10) {'BENIGN': 1383, 'MALIGNANT': 932}
Val: (549, 10) {'BENIGN': 300, 'MALIGNANT': 249}
Test: (422, 10) {'BENIGN': 248, 'MALIGNANT': 174}


In [3]:
# Dataset: lee `image_path_local` del manifest y aplica transforms.
# Train: flip + rotación leve; val/test: solo resize + tensor + normalización ImageNet.
class MammographyDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path_local"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label


IMG_SIZE = 224
BATCH_SIZE = 16

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = MammographyDataset(tr_df, transform=train_tfms)
val_ds = MammographyDataset(val_df, transform=eval_tfms)
test_ds = MammographyDataset(test_df, transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [4]:
# Backbone ResNet18 con pesos ImageNet; sustituimos `fc` por clasificador binario.
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
# Entrenamiento: varias épocas; guardamos el state_dict con mejor accuracy en validación.
def run_epoch(loader, training: bool):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    with torch.set_grad_enabled(training):
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            if training:
                optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * y.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total


EPOCHS = 8
best_val_acc = 0.0
best_path = ROOT / "reports/models/resnet18_best.pt"
best_path.parent.mkdir(parents=True, exist_ok=True)

history = []
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    va_loss, va_acc = run_epoch(val_loader, training=False)

    history.append((epoch, tr_loss, tr_acc, va_loss, va_acc))
    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), best_path)

print("Best val acc:", round(best_val_acc, 4))
print("Best model:", best_path)

Epoch 01 | train_loss=0.6605 train_acc=0.6285 | val_loss=0.6462 val_acc=0.6375
Epoch 02 | train_loss=0.5889 train_acc=0.6842 | val_loss=0.6960 val_acc=0.5993
Epoch 03 | train_loss=0.5559 train_acc=0.7114 | val_loss=0.7101 val_acc=0.6248
Epoch 04 | train_loss=0.5081 train_acc=0.7348 | val_loss=0.6581 val_acc=0.6448
Epoch 05 | train_loss=0.4887 train_acc=0.7603 | val_loss=0.6571 val_acc=0.6047
Epoch 06 | train_loss=0.4582 train_acc=0.7793 | val_loss=0.7203 val_acc=0.6375
Epoch 07 | train_loss=0.4360 train_acc=0.7797 | val_loss=0.6865 val_acc=0.6302
Epoch 08 | train_loss=0.4249 train_acc=0.8039 | val_loss=0.6741 val_acc=0.6102
Best val acc: 0.6448
Best model: G:\Cosas_programacion\Breast Cancer Interpretable-ml\reports\models\resnet18_best.pt


In [6]:
# Evaluación en conjunto de test (hold-out del manifest) con el mejor checkpoint.
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(dim=1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(y.numpy().tolist())

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["BENIGN", "MALIGNANT"]))

Confusion matrix:
[[149  99]
 [ 76  98]]

Classification report:
              precision    recall  f1-score   support

      BENIGN       0.66      0.60      0.63       248
   MALIGNANT       0.50      0.56      0.53       174

    accuracy                           0.59       422
   macro avg       0.58      0.58      0.58       422
weighted avg       0.59      0.59      0.59       422



## Posibles siguientes pasos
- Añadir ROC-AUC y PR-AUC.
- Probar pesos de clase o muestreo balanceado (ver notebook `02_resnet18_tuned.ipynb`).
- Ajustar tamaño de imagen, batch y learning rate.
- Interpretabilidad: Grad-CAM o flujo KAN en notebooks `04`–`08`.